# CardioIA — Fase 4 | Parte 2: Classificação de Imagens Médicas com CNN

Este notebook treina e compara **duas abordagens** de classificação de raios-X de
tórax (NORMAL vs PNEUMONIA):

1. **CNN simples treinada do zero** — arquitetura própria;
2. **Transfer Learning com VGG16** — modelo pré-treinado no ImageNet.

Ambas são avaliadas com **acurácia, matriz de confusão, precision, recall e F1-score**,
e ao final há um **protótipo interativo** para classificar novas imagens.

> ⚙️ **Como executar:** Google Colab → `Ambiente de execução > Alterar tipo de ambiente > GPU (T4)`
> Tempo estimado de treino: ~15-25 min no total.

## 1. Setup — pipeline de pré-processamento (Parte 1)

Reproduzimos aqui as etapas definidas e justificadas no notebook
`01_preprocessamento.ipynb`, garantindo que ambos os modelos recebam exatamente os
mesmos dados.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VAL_SPLIT = 0.20
EPOCHS = 15

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
DATA_DIR = Path(dataset_path) / "chest_xray"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=VAL_SPLIT, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=VAL_SPLIT, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False,
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
# Normalização + augmentation (mesmas decisões da Parte 1)
normalization = tf.keras.layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
])

AUTOTUNE = tf.data.AUTOTUNE


def preparar(ds, augment=False, shuffle=False):
    ds = ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_prep = preparar(train_ds, augment=True, shuffle=True)
val_prep = preparar(val_ds)
test_prep = preparar(test_ds)

# Pesos de classe para compensar o desbalanceamento (~3:1)
n_normal = len(list((DATA_DIR / "train" / "NORMAL").glob("*.jpeg")))
n_pneu = len(list((DATA_DIR / "train" / "PNEUMONIA").glob("*.jpeg")))
n_total = n_normal + n_pneu
class_weight = {0: n_total / (2.0 * n_normal), 1: n_total / (2.0 * n_pneu)}
print("Pesos de classe:", class_weight)

## 2. Funções auxiliares de avaliação

Centralizamos a avaliação numa única função para garantir que **as duas abordagens
sejam medidas exatamente da mesma forma** — requisito para uma comparação justa.

In [ ]:
def plotar_historico(history, titulo):
    """Curvas de acurácia e perda (treino vs validação)."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history["accuracy"], label="treino")
    axes[0].plot(history.history["val_accuracy"], label="validação")
    axes[0].set_title(f"{titulo} — Acurácia")
    axes[0].set_xlabel("Época")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="treino")
    axes[1].plot(history.history["val_loss"], label="validação")
    axes[1].set_title(f"{titulo} — Perda")
    axes[1].set_xlabel("Época")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def avaliar_modelo(model, test_ds_prep, titulo):
    """Avalia no conjunto de teste: acurácia, matriz de confusão e
    classification_report (precision, recall, F1)."""
    y_true = np.concatenate([y.numpy() for _, y in test_ds_prep]).ravel()
    y_prob = model.predict(test_ds_prep, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = (y_pred == y_true).mean()
    print(f"\n{'=' * 55}\n{titulo}\n{'=' * 55}")
    print(f"Acurácia no teste: {acc:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Matriz de Confusão — {titulo}")
    plt.show()

    rel = classification_report(y_true, y_pred, target_names=class_names,
                                output_dict=True)
    return {
        "titulo": titulo,
        "acuracia": acc,
        "precision_pneumonia": rel["PNEUMONIA"]["precision"],
        "recall_pneumonia": rel["PNEUMONIA"]["recall"],
        "f1_pneumonia": rel["PNEUMONIA"]["f1-score"],
        "recall_normal": rel["NORMAL"]["recall"],
        "f1_macro": rel["macro avg"]["f1-score"],
    }

## 3. Abordagem 1 — CNN simples treinada do zero

Arquitetura clássica com **4 blocos convolucionais** (Conv2D + MaxPooling), seguidos
de camadas densas com Dropout:

- Filtros crescentes (32→64→128→128): as primeiras camadas detectam padrões simples
  (bordas, texturas) e as mais profundas combinam-nos em estruturas complexas
  (opacidades pulmonares, contornos);
- `Dropout(0.5)` na cabeça densa para reduzir overfitting;
- Saída `sigmoid` (1 neurônio) por ser classificação **binária**.

In [ ]:
def construir_cnn_do_zero():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name="cnn_do_zero")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn = construir_cnn_do_zero()
cnn.summary()

In [ ]:
callbacks_cnn = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

hist_cnn = cnn.fit(
    train_prep,
    validation_data=val_prep,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks_cnn,
)

In [ ]:
plotar_historico(hist_cnn, "CNN do zero")
resultado_cnn = avaliar_modelo(cnn, test_prep, "CNN do zero")

## 4. Abordagem 2 — Transfer Learning com VGG16

A **VGG16** foi pré-treinada no ImageNet (1,4 milhão de imagens). Suas camadas
convolucionais já sabem extrair características visuais genéricas (bordas, texturas,
formas). A estratégia tem **duas fases**:

1. **Feature extraction:** congelamos toda a base VGG16 e treinamos apenas uma
   "cabeça" densa nova para o nosso problema;
2. **Fine-tuning:** descongelamos o último bloco convolucional (`block5`) e
   re-treinamos com learning rate bem menor (1e-5), adaptando as características de
   mais alto nível ao domínio de raios-X.

> Nota: a VGG16 espera entradas pré-processadas pelo `preprocess_input` próprio
> (subtração da média do ImageNet, canais BGR) — por isso usamos o dataset **sem**
> a normalização /255, aplicando o pré-processamento oficial dentro do modelo.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# Datasets para a VGG16: sem Rescaling(1/255); o preprocess_input cuida da escala
def preparar_vgg(ds, augment=False, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_vgg = preparar_vgg(train_ds, augment=True, shuffle=True)
val_vgg = preparar_vgg(val_ds)
test_vgg = preparar_vgg(test_ds)

In [ ]:
base_vgg = VGG16(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_vgg.trainable = False  # Fase 1: base congelada

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = preprocess_input(inputs)
x = base_vgg(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

modelo_tl = tf.keras.Model(inputs, outputs, name="vgg16_transfer_learning")
modelo_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
modelo_tl.summary()

In [ ]:
# Fase 1 — feature extraction (base congelada)
hist_tl = modelo_tl.fit(
    train_vgg,
    validation_data=val_vgg,
    epochs=8,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True)],
)

In [ ]:
# Fase 2 — fine-tuning do último bloco convolucional com LR reduzido
base_vgg.trainable = True
for layer in base_vgg.layers:
    layer.trainable = layer.name.startswith("block5")

modelo_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR 10x menor: ajuste fino
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

hist_ft = modelo_tl.fit(
    train_vgg,
    validation_data=val_vgg,
    epochs=5,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True)],
)

In [ ]:
plotar_historico(hist_tl, "VGG16 — feature extraction")
plotar_historico(hist_ft, "VGG16 — fine-tuning")
resultado_tl = avaliar_modelo(modelo_tl, test_vgg, "Transfer Learning (VGG16)")

## 5. Comparação das duas abordagens

In [ ]:
import pandas as pd

comparacao = pd.DataFrame([resultado_cnn, resultado_tl]).set_index("titulo")
comparacao.round(4)

**Como interpretar (contexto médico):**

- **Recall de PNEUMONIA** é a métrica mais crítica: mede a fração de casos doentes
  corretamente identificados. Um **falso negativo** (paciente doente classificado
  como normal) é o erro mais perigoso em triagem;
- **Recall de NORMAL** indica quantos saudáveis são corretamente liberados — recall
  baixo aqui gera excesso de falsos alarmes e sobrecarga clínica;
- **F1 macro** equilibra as duas classes e é mais confiável que a acurácia neste
  dataset desbalanceado (~3:1).

Em geral, espera-se que o **Transfer Learning supere a CNN do zero**, pois reaproveita
características visuais aprendidas em milhões de imagens — com menos épocas de treino.

## 6. Salvando o melhor modelo

O modelo salvo (`.keras`) é consumido pelo protótipo Flask (pasta `app/` do repositório).

In [ ]:
melhor = modelo_tl if resultado_tl["f1_macro"] >= resultado_cnn["f1_macro"] else cnn
print("Melhor modelo:", melhor.name)
melhor.save("modelo_cardioia.keras")
print("Salvo em modelo_cardioia.keras")

# Opcional: copiar para o Google Drive para não perder ao fim da sessão
# from google.colab import drive
# drive.mount('/content/drive')
!cp modelo_cardioia.keras /content/drive/MyDrive/

## 7. Protótipo interativo — classifique uma imagem

Execute a célula abaixo, envie um raio-X de tórax (JPEG/PNG) e veja a predição do
modelo com o nível de confiança. Este é o protótipo de apresentação dos resultados
em notebook interativo (há também uma versão web em Flask no repositório).

In [ ]:
from google.colab import files
from PIL import Image as PILImage


def classificar_imagem(caminho, modelo, usa_preprocess_vgg=True):
    img = PILImage.open(caminho).convert("RGB").resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32)[np.newaxis, ...]
    if not usa_preprocess_vgg:          # CNN do zero espera [0,1]
        arr = arr / 255.0
    prob = float(modelo.predict(arr, verbose=0).ravel()[0])
    classe = class_names[int(prob >= 0.5)]
    confianca = prob if prob >= 0.5 else 1 - prob

    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    cor = "#e53935" if classe == "PNEUMONIA" else "#4caf50"
    plt.title(f"{classe} — confiança {confianca:.1%}", color=cor, fontsize=14)
    plt.axis("off")
    plt.show()

    if classe == "PNEUMONIA":
        print("⚠️  Padrão sugestivo de pneumonia detectado. "
              "Encaminhar para avaliação médica.")
    else:
        print("✅ Nenhum padrão anormal detectado pelo modelo.")
    print("\n⚕️ AVISO: protótipo educacional — NÃO substitui diagnóstico médico.")


uploaded = files.upload()
for nome in uploaded:
    classificar_imagem(nome, melhor, usa_preprocess_vgg=(melhor is modelo_tl))

## 8. Conclusão

Este notebook cumpriu os requisitos da Parte 2:

1. ✅ **CNN do zero** implementada e treinada (4 blocos convolucionais);
2. ✅ **Transfer Learning com VGG16** em duas fases (feature extraction + fine-tuning);
3. ✅ **Avaliação completa**: acurácia, matriz de confusão, precision, recall e F1;
4. ✅ **Protótipo interativo** de classificação (este notebook) + app Flask no repositório.

**Limitações e responsabilidade:** o modelo foi treinado num único dataset, sem
metadados demográficos, e não passou por validação clínica. Trata-se de um protótipo
educacional de apoio à decisão — nunca um substituto do diagnóstico profissional.
A discussão completa de vieses e fairness está no relatório do Ir Além 1.